# Notebook 2: Panel Construction

**Input:** `features_panel.csv`
**Output:** `model_ready.csv`
**Purpose:** Convert the daily feature panel to a weekly forecasting panel, apply
coverage filters, and define the development and out-of-sample periods used in modeling.

In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## 2.1 Weekly Sampling

The daily feature panel is converted to a weekly forecasting panel by selecting
one forecast origin per calendar week — the first available trading day of that week.
This handles Vietnamese market holidays naturally without hardcoding Monday as origin.

The sample window is restricted to **2014-01-01 → 2025-12-31** based on forecast-origin
date. Observations with missing target are dropped at this stage.

In [22]:
# 2.1.1 Load and restrict sample window
df = pd.read_csv("output/intermediate/features_panel.csv")
df["date"] = pd.to_datetime(df["date"])

df = df[
    (df["date"] >= "2014-01-01") &
    (df["date"] <= "2025-12-31")
].copy()

df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

In [23]:
# 2.1.2 Build weekly origins
# First available trading day of each ISO week

market_dates = pd.DataFrame({"date": sorted(df["date"].dropna().unique())})
iso = market_dates["date"].dt.isocalendar()
market_dates["iso_year"] = iso.year
market_dates["iso_week"] = iso.week

weekly_origins = (
    market_dates
    .groupby(["iso_year", "iso_week"], as_index=False)["date"]
    .min()
    .rename(columns={"date": "weekly_origin"})
)

weekly_dates = set(weekly_origins["weekly_origin"])

In [24]:
# 2.1.3 Filter to weekly panel and drop missing target
df_weekly = df[df["date"].isin(weekly_dates)].copy()
df_weekly = df_weekly[df_weekly["target_5d_mktadj"].notna()].copy()
df_weekly = df_weekly.sort_values(["ticker", "date"]).reset_index(drop=True)

In [25]:
# 2.1.4 Validation
print("Shape:          ", df_weekly.shape)
print("Tickers:        ", df_weekly["ticker"].nunique())
print("Date range:     ", df_weekly["date"].min().date(), "->", df_weekly["date"].max().date())
print("Weekly origins: ", df_weekly["date"].nunique())
print("Missing target: ", df_weekly["target_5d_mktadj"].isna().sum())

# Tickers in daily panel but missing from weekly panel
tickers_daily  = set(df["ticker"].dropna().unique())
tickers_weekly = set(df_weekly["ticker"].dropna().unique())
print("\nTickers dropped:", sorted(tickers_daily - tickers_weekly))

# Weekly coverage distribution
print("\nWeekly ticker coverage:")
print(df_weekly.groupby("date")["ticker"].nunique().describe())

Shape:           (114610, 35)
Tickers:         219
Date range:      2014-01-02 -> 2025-12-29
Weekly origins:  622
Missing target:  0

Tickers dropped: ['HAI', 'HVG']

Weekly ticker coverage:
count    622.000000
mean     184.260450
std       21.628015
min      132.000000
25%      162.000000
50%      195.000000
75%      200.000000
max      210.000000
Name: ticker, dtype: float64


## 2.2 Model-Ready Dataset

The weekly panel is filtered to the columns needed for modeling. Predictors are
organized into three nested blocks. Rows with any missing value across target or
predictors are dropped. Weeks with fewer than 100 usable tickers are removed to
avoid unstable forecasting periods.

The sample is split into two periods:
- **Development (2014–2015):** reserved for hyperparameter tuning
- **OOS (2016–2025):** reserved for expanding-window forecasting

In [26]:
# 2.2.1 Define target and feature blocks
TARGET_COL = "target_5d_mktadj"

block1_features = [
    "ret_1d_lag", "ret_5d_lag", "momentum_1m", "volatility_1m", "log_size"
]
block2_features = block1_features + ["turnover_5d", "amihud_5d"]
block3_features = block2_features + ["f_buy_5d", "f_sell_5d"]

feature_sets = {
    "b1": block1_features,
    "b2": block2_features,
    "b3": block3_features
}

In [27]:
# 2.2.2 Filter to model-ready columns and drop missing values
keep_cols = ["ticker", "date", TARGET_COL] + block3_features

df_model = df_weekly[keep_cols].dropna().copy()


# 2.2.3 Coverage filter
# Remove weeks with fewer than 100 tickers to avoid unstable forecasting periods
MIN_TICKERS = 100

weekly_coverage = (
    df_model.groupby("date")["ticker"]
    .nunique()
    .reset_index(name="n_tickers")
)

valid_weeks   = weekly_coverage[weekly_coverage["n_tickers"] >= MIN_TICKERS]["date"]
dropped_weeks = weekly_coverage[weekly_coverage["n_tickers"] <  MIN_TICKERS].copy()

df_model = df_model[df_model["date"].isin(valid_weeks)].copy()


# 2.2.4 Sample period flags
df_model["sample_period"] = "oos"
df_model.loc[df_model["date"] < "2016-01-01", "sample_period"] = "development"

df_model = df_model.sort_values(["ticker", "date"]).reset_index(drop=True)

In [28]:
# 2.2.5 Winsorization and rank normalization
# Applied after column selection and NaN drop, before saving

# Winsorize features and target at 1%/99% each week
# Clips extreme cross-sectional values; protects FE (OLS) from outlier influence
cols_to_winsorize = block3_features + [TARGET_COL]
df_model[cols_to_winsorize] = df_model.groupby("date")[cols_to_winsorize].transform(
    lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99))
)

# Save descriptive_ready for Notebook 5
# Post-winsorization, pre-rank-normalization: preserves original scale for descriptive statistics
df_model.to_csv("output/intermediate/descriptive_ready.csv", index=False)
print("Saved: output/intermediate/descriptive_ready.csv")

# Cross-sectional rank normalization of features only (not target)
# Following Gu et al. (2020): transform each feature to [-0.5, 0.5] within each week
# Target (target_5d_mktadj) is retained in its original scale to preserve economic meaning
def rank_normalize(x):
    r = x.rank(method="average", na_option="keep")
    n = r.count()
    if n <= 1:
        return x * 0
    return (r - 1) / (n - 1) - 0.5

df_model[block3_features] = df_model.groupby("date")[block3_features].transform(rank_normalize)

print("Winsorization + rank normalization applied.")
print(f"Target range after winsorization: {df_model[TARGET_COL].min():.4f} -> {df_model[TARGET_COL].max():.4f}")
print("Feature range after normalization (expected near [-0.5, 0.5]):")
print(df_model[block3_features].agg(["min", "max"]).round(4).to_string())

Saved: output/intermediate/descriptive_ready.csv
Winsorization + rank normalization applied.
Target range after winsorization: -0.3761 -> 0.2984
Feature range after normalization (expected near [-0.5, 0.5]):
     ret_1d_lag  ret_5d_lag  momentum_1m  volatility_1m  log_size  turnover_5d  amihud_5d  f_buy_5d  f_sell_5d
min     -0.4975     -0.4975      -0.4975        -0.4975   -0.4975      -0.4975    -0.4975   -0.4829    -0.4874
max      0.4975      0.4975       0.4975         0.4975    0.4975       0.4975     0.4975    0.4975     0.4975


In [29]:
# 2.2.6 Save and validation
df_model.to_csv("output/intermediate/model_ready.csv", index=False)

if len(dropped_weeks) > 0:
    dropped_weeks.to_csv("output/intermediate/dropped_weeks_log.csv", index=False)

dev = df_model[df_model["sample_period"] == "development"]
oos = df_model[df_model["sample_period"] == "oos"]

print("Shape:    ", df_model.shape)
print("Tickers:  ", df_model["ticker"].nunique())
print("Missing:  ", df_model.isna().sum().sum())

print("\nDevelopment:", dev["date"].min().date(), "->", dev["date"].max().date(),
      "| rows:", len(dev))
print("OOS:        ", oos["date"].min().date(), "->", oos["date"].max().date(),
      "| rows:", len(oos))

print("\nWeekly coverage:")
print(df_model.groupby("date")["ticker"].nunique().describe())

print("\nDropped weeks (< 100 tickers):")
print(dropped_weeks if len(dropped_weeks) > 0 else "None")

print("\nSaved: model_ready.csv |", df_model.shape)

Shape:     (111858, 13)
Tickers:   219
Missing:   0

Development: 2014-02-17 -> 2015-12-28 | rows: 13882
OOS:         2016-01-04 -> 2025-12-29 | rows: 97976

Weekly coverage:
count    610.000000
mean     183.373770
std       22.106281
min      131.000000
25%      161.000000
50%      195.000000
75%      200.000000
max      210.000000
Name: ticker, dtype: float64

Dropped weeks (< 100 tickers):
          date  n_tickers
37  2014-11-03          6
538 2024-07-08          3
560 2024-12-09          7
561 2024-12-16          2
563 2024-12-30          4

Saved: model_ready.csv | (111858, 13)
